# GPFL Server

> The core abstraction for `GPFL` server: [https://arxiv.org/pdf/2308.10279v3](https://arxiv.org/pdf/2308.10279v3)

In [ ]:
#| default_exp servers.gpfl

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os
import gc
import copy
from tqdm import tqdm
from loguru import logger

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from fedai.client_selector import BaseClientSelector
from fedai.data import prepare_dl
from fedai.models import create_model
from fedai.optimizers import get_optimizer

from fedai.servers.base_server import BaseServer
from fedai.utils.registery import AlgorithmRegistry
from fedai.core import get_clean_kwargs


In [ ]:
#| export
class GCE(nn.Module):
    def __init__(self, in_features, num_classes, device='cpu'):
        super(GCE, self).__init__()
        self.in_features = in_features
        self.num_classes = num_classes
        self.embedding = nn.Embedding(num_classes, in_features)
        self.device = device

    def forward(self, x, label):
        embeddings = self.embedding(torch.tensor(range(self.num_classes), device=self.device))
        cosine = F.linear(F.normalize(x), F.normalize(embeddings))
        one_hot = torch.zeros(cosine.size(), device=self.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)

        softmax_value = F.log_softmax(cosine, dim=1)
        softmax_loss = one_hot * softmax_value
        softmax_loss = - torch.mean(torch.sum(softmax_loss, dim=1))

        return softmax_loss


In [ ]:
#| hide
model = GCE(in_features=128, num_classes=10)
d = model.state_dict()
for key in d.keys():
    print(key)

embedding.weight


In [ ]:
#| export
class CoV(nn.Module):
    def __init__(self, in_dim):
        super(CoV, self).__init__()

        self.Conditional_gamma = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(),
            nn.LayerNorm([in_dim]),
        )
        self.Conditional_beta = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(),
            nn.LayerNorm([in_dim]),
        )
        self.act = nn.ReLU()

    def forward(self, x, context):
        gamma = self.Conditional_gamma(context)
        beta = self.Conditional_beta(context)

        out = torch.multiply(x, gamma + 1)
        out = torch.add(out, beta)
        out = self.act(out)
        return out

In [ ]:
#| export
@AlgorithmRegistry.register_server("gpfl")
class ServerGPFL(BaseServer):
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        super().__init__(cfg, client_selector, client_cls, criterion, fds, writer, device, **kwargs) 

        self.GCE = GCE(in_features=self.cfg.model.hidden_dim,
                       num_classes=self.cfg.data.num_classes,
                       device=self.device)
        
        self.CoV = CoV(self.cfg.model.hidden_dim)

In [ ]:
#| export
@patch
def client_fn(self: ServerGPFL, id, comm_round, client_state):

    if (comm_round == 1 and client_state == {}) or client_state == {}:
        client_state['model'] = copy.deepcopy(self.model.state_dict())
        client_state['GCE'] = copy.deepcopy(self.GCE.state_dict())
        client_state['CoV'] = copy.deepcopy(self.CoV.state_dict())
        client_state['GCE_frozen'] = copy.deepcopy(self.GCE.state_dict())
        client_state['generic_conditional_input'] = torch.zeros(self.cfg.model.hidden_dim)
        client_state['personalized_conditional_input'] = torch.zeros(self.cfg.model.hidden_dim)

    model = create_model(self.cfg)
    model.load_state_dict(client_state['model'])
    client_state['model'] = model

    inst_GCE = GCE(in_features=self.cfg.model.hidden_dim,
                                num_classes=self.cfg.data.num_classes,
                                device=self.device)
    inst_GCE.load_state_dict(client_state['GCE'])
    client_state['GCE'] = inst_GCE

    inst_CoV = CoV(in_dim=self.cfg.model.hidden_dim)
    inst_CoV.load_state_dict(client_state['CoV'])
    client_state['CoV'] = inst_CoV
    
    inst_GCE_frozen = GCE(in_features=self.cfg.model.hidden_dim,
                            num_classes=self.cfg.data.num_classes,
                            device=self.device)
    inst_GCE_frozen.load_state_dict(client_state['GCE_frozen'])
    client_state['GCE_frozen'] = inst_GCE_frozen
    
    kwargs = get_clean_kwargs(self.cfg.optimizer)
    kwargs.pop("cls", None)
    optimizer = get_optimizer(self.cfg)(model.parameters(), **kwargs)
    optimizer.load_state_dict(client_state['optimizer']) if 'optimizer' in client_state else None
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(self.device)

    client_state['optimizer'] = optimizer

    kwargs.pop("lr", None)
    GCE_opt = get_optimizer(self.cfg)(self.GCE.parameters(), lr= self.cfg.algorithm.gce_lr, **kwargs)
    GCE_opt.load_state_dict(client_state['GCE_optimizer']) if 'GCE_optimizer' in client_state else None
    for state in GCE_opt.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(self.device)
    client_state['GCE_optimizer'] = GCE_opt

    CoV_opt = get_optimizer(self.cfg)(self.CoV.parameters(), lr= self.cfg.algorithm.coV_lr, **kwargs)
    CoV_opt.load_state_dict(client_state['CoV_optimizer']) if 'CoV_optimizer' in client_state else None
    for state in CoV_opt.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(self.device)
    client_state['CoV_optimizer'] = CoV_opt

    train_loader = prepare_dl(self.cfg, id, self.fds, train=True, distributed=False)
    test_loader = prepare_dl(self.cfg, id, self.fds, train=False, distributed=False)
    client = self.client_cls(id= id, 
                             cfg= self.cfg,
                             train_loader= train_loader,
                             test_loader= test_loader,
                             state= client_state,
                             criterion= self.criterion,
                             device= self.device,
                             t= comm_round)
    client.logger = self.logger
    return client


### Aggregation


In [ ]:
#| export
@patch
def aggregate(self: ServerGPFL, lst_active_ids, comm_round, len_clients_ds):
    m_t = sum(len_clients_ds.values())
    
    with torch.no_grad():
        global_model = None
        
        for id in lst_active_ids:
            client_state_dict = self.state_mgr.get_state(id).get('model', None)

            if global_model is None:
                global_model = {k: torch.zeros_like(v) for k, v in client_state_dict.items()}

            n_k = len_clients_ds[id]
            weight = n_k / m_t
            for key in global_model.keys():
                global_model[key].add_(client_state_dict[key], alpha=weight)

        self.model.load_state_dict(global_model)
        
        for id in lst_active_ids:
            client_model = self.state_mgr.get_state(id).get('model', None)
            for key in global_model.keys():
                if key.startswith('backbone'):
                    self.logger.info(f"Updating client {id} model with global backbone parameters.")
                    client_model[key] = self.model.state_dict()[key]

            self.state_mgr.set_state(id, {
                    **self.state_mgr.get_state(id),
                    'model': client_model,
                })
            
    self.global_GCE(lst_active_ids, comm_round, len_clients_ds)
    self.global_CoV(lst_active_ids, comm_round, len_clients_ds)

In [ ]:
#| export
@patch
def global_GCE(self: ServerGPFL, lst_active_ids, comm_round, len_clients_ds):
    m_t = sum(len_clients_ds.values())        
    with torch.no_grad():
        global_model = None
        for id in lst_active_ids:
            client_GCE = self.state_mgr.get_state(id).get('GCE', None)

            if global_model is None:
                global_model = {k: torch.zeros_like(v) for k, v in client_GCE.items()}

            n_k = len_clients_ds[id]
            weight = n_k / m_t
            for key in global_model.keys():
                global_model[key].add_(client_GCE[key], alpha=weight)
        self.GCE.load_state_dict(global_model)

        for id in lst_active_ids:
            client_state = self.state_mgr.get_state(id)
            client_state['generic_conditional_input'] = torch.zeros(self.cfg.model.hidden_dim)
            client_state['personalized_conditional_input'] = torch.zeros(self.cfg.model.hidden_dim)

            embeddings = client_state['GCE']['embedding.weight'][torch.tensor(range(self.cfg.data.num_classes))]
            for l, emb in enumerate(embeddings):
                client_state['generic_conditional_input'].data += emb / self.cfg.data.num_classes
                client_state['personalized_conditional_input'].data += emb * client_state['sample_per_class'][l]

            for key in global_model.keys():
                self.logger.info(f"Updating client {id} GCE with global GCE parameters.")
                client_state['GCE'][key].copy_(self.GCE.state_dict()[key])

            client_state['GCE_frozen'] = copy.deepcopy(client_state['GCE'])

            self.state_mgr.set_state(id, client_state)

In [ ]:
#| export
@patch
def global_CoV(self: ServerGPFL, lst_active_ids, comm_round, len_clients_ds):
    m_t = sum(len_clients_ds.values())
    with torch.no_grad():
        global_model = None
        for id in lst_active_ids:
            client_CoV = self.state_mgr.get_state(id).get('CoV', None)

            if global_model is None:
                global_model = {k: torch.zeros_like(v) for k, v in client_CoV.items()}

            n_k = len_clients_ds[id]
            weight = n_k / m_t
            for key in global_model.keys():
                global_model[key].add_(client_CoV[key], alpha=weight)
        self.CoV.load_state_dict(global_model)

        for id in lst_active_ids:
            client_CoV = self.state_mgr.get_state(id).get('CoV', None)
            for key in global_model.keys():
                self.logger.info(f"Updating client {id} CoV with global CoV parameters.")
                client_CoV[key].copy_(self.CoV.state_dict()[key])

            client_state = {**self.state_mgr.get_state(id), 'CoV': client_CoV}
            self.state_mgr.set_state(id, client_state)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()